In [1]:
from yfinance import download
import numpy as np
import plotly.graph_objects as go

In [2]:
df = download("TAEE11.SA", period="10y", interval="1d", progress=False, auto_adjust=False).droplevel(1, axis=1)

In [3]:
window = 21

retorno = df["Adj Close"].pct_change()

x = retorno.rolling(window).std().dropna()
y = retorno.rolling(window).mean().dropna()

# Centro da elipse
media_x = x.mean()
media_y = y.mean()

# Matriz de covariância
cov = np.cov(x, y)

# Autovalores e autovetores
valores, vetores = np.linalg.eigh(cov)

# Ordena do maior para o menor
ordem = valores.argsort()[::-1]
valores = valores[ordem]
vetores = vetores[:, ordem]

# Ângulo da elipse
angulo = np.linspace(0, 2*np.pi, 200)

# Escala da elipse (95% confiança ~ 2 desvios)
escala = 2

elipse = (
    escala
    * np.sqrt(valores[:, None])
    * np.array([np.cos(angulo), np.sin(angulo)])
)

# Rotaciona
elipse_rot = vetores @ elipse

ellipse_x = media_x + elipse_rot[0]
ellipse_y = media_y + elipse_rot[1]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode="markers",
    name="Observações",
    marker=dict(size=6, opacity=0.5),
))

fig.add_trace(go.Scatter(
    x=[x.iloc[-1]],
    y=[y.iloc[-1]],
    mode="markers",
    name="Atual",
    marker=dict(size=12, color="black", symbol="diamond"),
))

fig.add_trace(go.Scatter(
    x=ellipse_x,
    y=ellipse_y,
    mode="lines",
    name="Elipse de confiança",
    line=dict(color="red")
))

fig.add_hline(y=y.mean(), line_dash="dot", line_color="gray")
fig.add_vline(x=x.mean(), line_dash="dot", line_color="gray")

fig.update_layout(
    template="plotly_white",
    title="Retorno vs Volatilidade",
    xaxis_title="Volatilidade",
    yaxis_title="Retorno",
)

fig.show()